# E8 (v2) — Per-class F1 cho 5 classifier — pipeline SAO CHÉP NGUYÊN VĂN từ notebook train

**Bài học từ v1 (mọi sanity check FAIL):** tập crop và preprocessing phải giống HỆT bản train, không được "tương đương".
v2 sao chép nguyên văn từ notebook 01/02/03:

1. **build_crops** đúng nguyên bản: `pad=0.15` quanh box, bỏ box `w<16 or h<16` px, trần `MAX_CROPS_PER_CLASS=2500`
   (không chạm với test), duyệt file `sorted` rồi `random.Random(42).shuffle`, crop ở size **128** (PIL resize mặc định).
2. **VGG16/ResNet50**: `keras.applications.<backbone>.preprocess_input(X.astype("float32"))` — kiểu caffe, KHÔNG phải /255.
3. **GRU**: `X/255.` rồi reshape `(N, 128, 384)` — như notebook 02.
4. **ViT-B/S**: từ crop 128 uint8 → `cv2.resize` lên 224 (bilinear) → `/255` → normalize ImageNet (0.485/0.229) — đúng CropDS của notebook 03.
5. **Metric** dùng đúng hàm `multiclass_metrics` của notebook (worst-F1 loại lớp support<10).

**Sanity check bắt buộc:** acc/macro-F1 phải khớp số công bố (ViT-B 0,9521/0,8711; ViT-S 0,9453/0,8429;
ResNet50 0,9014/0,8072; VGG16 0,8966/0,7894; GRU-256 0,8075/0,6780) trong dung sai ±0,5 điểm acc / ±2 điểm F1.
KHỚP hết → per-class F1 hợp lệ để đưa vào báo cáo.

**Đầu ra:** `results/e8_perclass_f1.csv`, `results/e8_confusion_<model>.png`, `results/e8_summary_check.csv`.

**Add Input:** `ppe-unified-v1` + `ppe-trained-weights`. GPU T4 khuyến nghị (~20 phút; CPU ~1-2h, ViT chậm nhất).

In [ ]:
KEY_PREFIX = "e8_"
# ==================== CONFIG (đúng hằng số của notebook train) ====================
QUICK_TEST = False
SEED = 42
CROP_SIZE, CROP_SIZE_VIT = 128, 224
MAX_CROPS_PER_CLASS = 200 if QUICK_TEST else 2500

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) and isinstance(_s[k], dict) and _s[k].get("v") == 2 for k in _s):
        STATE.update(_s); print("Resumed state (v2):", list(_s))
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q timm
import torch, numpy as np, cv2
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (chạy được, ViT sẽ chậm)")


In [ ]:
from pathlib import Path
import yaml
def find_dataset_root():
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Chưa Add Input dataset ppe-unified-v1!")
DATA = find_dataset_root()
CLASSES = yaml.safe_load(open(f"{DATA}/data.yaml"))["names"]
if isinstance(CLASSES, dict): CLASSES = [CLASSES[k] for k in sorted(CLASSES)]
NUM_CLS = len(CLASSES)
print("Dataset:", DATA, "|", NUM_CLS, "lớp")


In [ ]:
# ---------- build_crops: SAO CHÉP NGUYÊN VĂN từ notebook 01/02/03 (cell 5) ----------
from collections import Counter
from PIL import Image

def build_crops(split, size):
    X, y = [], []; per_class = Counter()
    files = sorted(os.listdir(f"{DATA}/{split}/labels"))
    random.Random(SEED).shuffle(files)
    for lf in files:
        stem = Path(lf).stem
        if "_rfs" in stem: continue
        for ext in [".png", ".jpg", ".jpeg"]:
            ip = f"{DATA}/{split}/images/{stem}{ext}"
            if os.path.exists(ip): break
        else: continue
        img = Image.open(ip).convert("RGB"); W, H = img.size
        for line in open(f"{DATA}/{split}/labels/{lf}"):
            if not line.strip(): continue
            c, xc, yc, bw, bh = map(float, line.split()); c = int(c)
            if per_class[c] >= MAX_CROPS_PER_CLASS: continue
            w, h = bw * W, bh * H
            if w < 16 or h < 16: continue
            pad = 0.15
            box = (max(0, (xc - bw/2) * W - pad * w), max(0, (yc - bh/2) * H - pad * h),
                   min(W, (xc + bw/2) * W + pad * w), min(H, (yc + bh/2) * H + pad * h))
            X.append(np.asarray(img.crop(box).resize((size, size)), dtype=np.uint8)); y.append(c)
            per_class[c] += 1
    return np.array(X), np.array(y)

Xte, yte = build_crops("test", CROP_SIZE)
print("Crop test:", Xte.shape, {CLASSES[i]: int(v) for i, v in enumerate(np.bincount(yte, minlength=NUM_CLS))})

# ---------- multiclass_metrics: nguyên văn từ notebook ----------
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
def multiclass_metrics(y_true, pred):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average="macro")
    per_cls = f1_score(y_true, pred, average=None, labels=list(range(NUM_CLS)), zero_division=0)
    support = np.bincount(y_true, minlength=NUM_CLS)
    valid = [c for c in range(NUM_CLS) if support[c] >= 10]
    worst = float(np.min([per_cls[c] for c in valid])) if valid else 0.0
    return acc, macro, worst


In [ ]:
# ---------- Tìm trọng số ----------
def find_one(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
        if hits: return hits[0]
    return None

WEIGHTS = {
    "ViT-B/16": find_one(["vit_base*best*.pt"]),
    "ViT-S/16": find_one(["vit_small*best*.pt"]),
    "ResNet50": find_one(["resnet50*best*.keras"]),
    "VGG16":    find_one(["vgg16*best*.keras"]),
    "GRU-256":  find_one(["rnn_best*.keras", "gru*best*.keras"]),
}
for k, v in WEIGHTS.items(): print(f"{k:10s} -> {v}")
PUBLISHED = {"ViT-B/16": (0.9521, 0.8711), "ViT-S/16": (0.9453, 0.8429),
             "ResNet50": (0.9014, 0.8072), "VGG16": (0.8966, 0.7894), "GRU-256": (0.8075, 0.6780)}


In [ ]:
# ---------- Predict theo ĐÚNG preprocessing từng nhánh ----------
import pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

def predict_keras_cnn(path, backbone):
    """VGG16/ResNet50: preprocess_input kiểu caffe — NGUYÊN VĂN notebook 01."""
    import tensorflow as tf
    from tensorflow import keras
    pp = {"VGG16": keras.applications.vgg16.preprocess_input,
          "ResNet50": keras.applications.resnet50.preprocess_input}[backbone]
    model = keras.models.load_model(path, compile=False)
    return model.predict(pp(Xte.astype("float32")), batch_size=64, verbose=0).argmax(1)

def predict_gru(path):
    """GRU: /255 rồi reshape (N, 128, 384) — NGUYÊN VĂN notebook 02."""
    from tensorflow import keras
    Xte_s = (Xte.astype("float32") / 255.).reshape(len(Xte), CROP_SIZE, CROP_SIZE * 3)
    model = keras.models.load_model(path, compile=False)
    return model.predict(Xte_s, batch_size=64, verbose=0).argmax(1)

def predict_vit(path, arch):
    """ViT: crop 128 uint8 -> cv2.resize 224 -> /255 -> normalize ImageNet — NGUYÊN VĂN CropDS notebook 03."""
    import timm
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    model = timm.create_model(arch, num_classes=NUM_CLS)
    model.load_state_dict(torch.load(path, map_location=dev)); model.to(dev).eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xte), 64):
            batch = []
            for img in Xte[i:i+64]:
                t = torch.from_numpy(cv2.resize(img, (CROP_SIZE_VIT, CROP_SIZE_VIT))).permute(2, 0, 1).float() / 255.
                batch.append((t - MEAN) / STD)
            preds.append(model(torch.stack(batch).to(dev)).argmax(1).cpu().numpy())
    return np.concatenate(preds)

rows, checks = [], []
for name, path in WEIGHTS.items():
    if path is None: print("BỎ QUA (thiếu weights):", name); continue
    if done(f"e8_{name}"): print("skip:", name); continue
    t0 = time.time()
    if name == "ViT-B/16": y_pred = predict_vit(path, "vit_base_patch16_224")
    elif name == "ViT-S/16": y_pred = predict_vit(path, "vit_small_patch16_224")
    elif name == "GRU-256": y_pred = predict_gru(path)
    else: y_pred = predict_keras_cnn(path, name)
    acc, mf1, worst = multiclass_metrics(yte, y_pred)
    pa, pf = PUBLISHED[name]
    ok = abs(acc - pa) < 0.005 and abs(mf1 - pf) < 0.02
    print(f"{name}: acc {acc:.4f} (công bố {pa}) | macro-F1 {mf1:.4f} ({pf}) | worst {worst:.4f} -> {'KHỚP ✓' if ok else 'LỆCH ✗'}")
    checks.append({"model": name, "acc": round(acc, 4), "acc_published": pa,
                   "macro_f1": round(mf1, 4), "f1_published": pf, "worst_f1": round(worst, 4), "sanity_ok": ok,
                   "min": round((time.time() - t0) / 60, 1)})
    rep = classification_report(yte, y_pred, labels=list(range(NUM_CLS)), target_names=CLASSES,
                                output_dict=True, zero_division=0)
    for cls in CLASSES:
        rows.append({"model": name, "class": cls,
                     **{k: round(rep[cls][k], 4) for k in ("precision", "recall", "f1-score")},
                     "support": int(rep[cls]["support"]), "sanity_ok": ok})
    cm = confusion_matrix(yte, y_pred, labels=list(range(NUM_CLS)), normalize="true")
    fig, ax = plt.subplots(figsize=(9, 8)); im_ = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLS)); ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
    ax.set_yticks(range(NUM_CLS)); ax.set_yticklabels(CLASSES, fontsize=7)
    ax.set_title(f"Confusion matrix (normalized) — {name} (test)"); fig.colorbar(im_)
    fig.savefig(f"{WORK}/results/e8_confusion_{name.replace('/', '_').replace('-', '_')}.png",
                dpi=120, bbox_inches="tight"); plt.close(fig)
    STATE[f"e8_{name}"] = {"v": 2, "acc": round(acc, 4), "macro_f1": round(mf1, 4), "sanity_ok": ok}; save_state()

pd.DataFrame(rows).to_csv(f"{WORK}/results/e8_perclass_f1.csv", index=False)
pd.DataFrame(checks).to_csv(f"{WORK}/results/e8_summary_check.csv", index=False)
print(pd.DataFrame(checks).to_string(index=False))


In [ ]:
# ==================== DỌN OUTPUT ====================
KEEP = {"results", "state.json"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Output cuối:", sorted(os.listdir(WORK)))
